# SAHI — Boost du rappel YOLO sur petits objets

SAHI (Slicing Aided Hyper Inference) découpe chaque image en tuiles chevauchantes,
fait tourner YOLO sur chaque tuile, puis fusionne les détections.
Les méduses qui font 0.25–1.5 % de l'image deviennent bien plus grandes
dans chaque tuile 640×640 → le rappel augmente mécaniquement.

**Structure du notebook :**
1. Installation
2. Configuration (tous les chemins ici)
3. Montage Drive + vérification
4. Fonctions utilitaires (IoU, matching GT)
5. Baseline YOLOv8 (métriques de référence)
6. Inférence SAHI (mêmes métriques)
7. Comparaison visuelle
8. Sweep des paramètres de tuile
9. Export vidéo annotée (optionnel)

In [ ]:
# ── 1. INSTALLATION ──────────────────────────────────────────
!pip install -q sahi ultralytics

In [ ]:
# ── 2. CONFIGURATION ─────────────────────────────────────────
# Adapter ces chemins selon ta structure Drive
DRIVE_ROOT      = "/content/drive/MyDrive"

# Poids du modèle entraîné
MODEL_PATH      = f"{DRIVE_ROOT}/runs/detect/meduse_detector/weights/best.pt"

# Images et labels de validation (format YOLO : classe cx cy w h normalisés)
VAL_IMAGES_DIR  = f"{DRIVE_ROOT}/dataset/val/images"
VAL_LABELS_DIR  = f"{DRIVE_ROOT}/dataset/val/labels"

# Vidéo pour export annoté (cellule 9, optionnel)
VIDEO_PATH      = f"{DRIVE_ROOT}/videos/ta_video.mp4"
VIDEO_OUT_PATH  = f"{DRIVE_ROOT}/videos/ta_video_sahi.mp4"

# ── Paramètres SAHI (point de départ, le sweep les optimisera) ──
SLICE_H         = 640   # hauteur d'une tuile en pixels
SLICE_W         = 640   # largeur d'une tuile en pixels
OVERLAP_RATIO   = 0.2   # chevauchement entre tuiles (0.0 → 0.5)

# ── Paramètres d'évaluation ──────────────────────────────────
CONF_THRESHOLD  = 0.25  # seuil confiance pour les prédictions
IOU_THRESHOLD   = 0.5   # seuil IoU pour matcher une prédiction à un GT

In [ ]:
# ── 3. MONTAGE DRIVE + VÉRIFICATION ──────────────────────────
from google.colab import drive
import os

drive.mount("/content/drive")

checks = [
    (MODEL_PATH,     "Modèle YOLO"),
    (VAL_IMAGES_DIR, "Dossier images val"),
    (VAL_LABELS_DIR, "Dossier labels val"),
]
all_ok = True
for path, label in checks:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗"
    print(f"{status} {label} : {path}")
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Corrige les chemins dans la cellule CONFIG avant de continuer.")

from pathlib import Path
n_images = len([p for p in Path(VAL_IMAGES_DIR).iterdir()
                if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
print(f"\n{n_images} images dans le set de validation.")

In [ ]:
# ── 4. FONCTIONS UTILITAIRES ──────────────────────────────────
import numpy as np
import cv2
from pathlib import Path


def get_image_files(folder):
    exts = {".jpg", ".jpeg", ".png"}
    return sorted([p for p in Path(folder).iterdir() if p.suffix.lower() in exts])


def load_yolo_labels(label_path, img_w, img_h):
    """Charge les GT YOLO et retourne une liste de [x1, y1, x2, y2] en pixels."""
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x1 = (cx - w / 2) * img_w
            y1 = (cy - h / 2) * img_h
            x2 = (cx + w / 2) * img_w
            y2 = (cy + h / 2) * img_h
            boxes.append([x1, y1, x2, y2])
    return boxes


def iou(a, b):
    xi1, yi1 = max(a[0], b[0]), max(a[1], b[1])
    xi2, yi2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, xi2 - xi1) * max(0.0, yi2 - yi1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def match_predictions(preds, gts, iou_thresh=None):
    """Retourne (TP, FP, FN) pour une image."""
    if iou_thresh is None:
        iou_thresh = IOU_THRESHOLD
    matched = set()
    tp = fp = 0
    for p in preds:
        best_iou, best_i = 0.0, -1
        for i, g in enumerate(gts):
            if i not in matched:
                v = iou(p, g)
                if v > best_iou:
                    best_iou, best_i = v, i
        if best_iou >= iou_thresh and best_i >= 0:
            tp += 1
            matched.add(best_i)
        else:
            fp += 1
    fn = len(gts) - len(matched)
    return tp, fp, fn


def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


print("Fonctions utilitaires chargées.")

In [ ]:
# ── 5. BASELINE YOLOv8 ────────────────────────────────────────
from ultralytics import YOLO

yolo = YOLO(MODEL_PATH)
images = get_image_files(VAL_IMAGES_DIR)

tp_base = fp_base = fn_base = 0
print(f"Évaluation baseline sur {len(images)} images...")

for img_path in images:
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    res = yolo(str(img_path), conf=CONF_THRESHOLD, verbose=False)[0]
    preds = (
        [[float(b[0]), float(b[1]), float(b[2]), float(b[3])]
         for b in res.boxes.xyxy.cpu().numpy()]
        if res.boxes is not None and len(res.boxes) > 0 else []
    )

    label_path = Path(VAL_LABELS_DIR) / (img_path.stem + ".txt")
    gts = load_yolo_labels(str(label_path), w, h)

    tp, fp, fn = match_predictions(preds, gts)
    tp_base += tp
    fp_base += fp
    fn_base += fn

p_base, r_base, f1_base = compute_metrics(tp_base, fp_base, fn_base)
print(f"\n{'='*40}")
print(f"  BASELINE YOLOv8  (conf={CONF_THRESHOLD})")
print(f"{'='*40}")
print(f"  TP={tp_base}  FP={fp_base}  FN={fn_base}")
print(f"  Precision = {p_base:.3f}")
print(f"  Recall    = {r_base:.3f}")
print(f"  F1        = {f1_base:.3f}")

In [ ]:
# ── 6. INFÉRENCE SAHI ─────────────────────────────────────────
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

sahi_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    confidence_threshold=CONF_THRESHOLD,
    device="cuda:0",
)

tp_sahi = fp_sahi = fn_sahi = 0
print(f"Évaluation SAHI (tuile={SLICE_H}×{SLICE_W}, overlap={OVERLAP_RATIO}) sur {len(images)} images...")

for img_path in images:
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    result = get_sliced_prediction(
        str(img_path),
        sahi_model,
        slice_height=SLICE_H,
        slice_width=SLICE_W,
        overlap_height_ratio=OVERLAP_RATIO,
        overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )
    preds = [
        [p.bbox.minx, p.bbox.miny, p.bbox.maxx, p.bbox.maxy]
        for p in result.object_prediction_list
    ]

    label_path = Path(VAL_LABELS_DIR) / (img_path.stem + ".txt")
    gts = load_yolo_labels(str(label_path), w, h)

    tp, fp, fn = match_predictions(preds, gts)
    tp_sahi += tp
    fp_sahi += fp
    fn_sahi += fn

p_sahi, r_sahi, f1_sahi = compute_metrics(tp_sahi, fp_sahi, fn_sahi)
print(f"\n{'='*40}")
print(f"  SAHI  (tuile={SLICE_H}×{SLICE_W}, overlap={OVERLAP_RATIO})")
print(f"{'='*40}")
print(f"  TP={tp_sahi}  FP={fp_sahi}  FN={fn_sahi}")
print(f"  Precision = {p_sahi:.3f}")
print(f"  Recall    = {r_sahi:.3f}")
print(f"  F1        = {f1_sahi:.3f}")
print(f"\n── Delta vs baseline ──")
print(f"  ΔRecall    = {r_sahi - r_base:+.3f}")
print(f"  ΔPrecision = {p_sahi - p_base:+.3f}")
print(f"  ΔF1        = {f1_sahi - f1_base:+.3f}")

In [ ]:
# ── 7. COMPARAISON VISUELLE ───────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
from PIL import Image, ImageDraw


def draw_boxes_pil(img_pil, boxes, color, width=2):
    draw = ImageDraw.Draw(img_pil)
    for b in boxes:
        draw.rectangle([(b[0], b[1]), (b[2], b[3])], outline=color, width=width)
    return img_pil


# ── Graphique métriques ──
fig, (ax_bar, ax_diff) = plt.subplots(1, 2, figsize=(12, 4))

metric_labels = ["Precision", "Recall", "F1"]
base_vals = [p_base, r_base, f1_base]
sahi_vals = [p_sahi, r_sahi, f1_sahi]
x = range(len(metric_labels))

bars1 = ax_bar.bar([i - 0.2 for i in x], base_vals, 0.38, label="Baseline", color="steelblue")
bars2 = ax_bar.bar([i + 0.2 for i in x], sahi_vals, 0.38, label="SAHI",     color="darkorange")
ax_bar.set_xticks(list(x))
ax_bar.set_xticklabels(metric_labels)
ax_bar.set_ylim(0, 1.1)
ax_bar.set_title("Baseline vs SAHI")
ax_bar.legend()
for b in list(bars1) + list(bars2):
    ax_bar.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.02,
                f"{b.get_height():.2f}", ha="center", va="bottom", fontsize=9)

deltas = [p_sahi - p_base, r_sahi - r_base, f1_sahi - f1_base]
colors = ["green" if d >= 0 else "red" for d in deltas]
ax_diff.bar(metric_labels, deltas, color=colors)
ax_diff.axhline(0, color="black", linewidth=0.8)
ax_diff.set_title("Delta SAHI − Baseline")
ax_diff.set_ylim(-0.3, 0.3)
for i, d in enumerate(deltas):
    ax_diff.text(i, d + (0.01 if d >= 0 else -0.02), f"{d:+.3f}",
                 ha="center", va="bottom" if d >= 0 else "top", fontsize=10)

plt.tight_layout()
plt.show()

# ── Visualisation sur 3 frames aléatoires ──
sample = random.sample(images, min(3, len(images)))
fig, axes = plt.subplots(len(sample), 2, figsize=(16, 6 * len(sample)))
if len(sample) == 1:
    axes = [axes]

for idx, img_path in enumerate(sample):
    img_arr = cv2.imread(str(img_path))
    h, w = img_arr.shape[:2]
    img_rgb = cv2.cvtColor(img_arr, cv2.COLOR_BGR2RGB)

    label_path = Path(VAL_LABELS_DIR) / (img_path.stem + ".txt")
    gts = load_yolo_labels(str(label_path), w, h)

    # Baseline
    res_b = yolo(str(img_path), conf=CONF_THRESHOLD, verbose=False)[0]
    preds_b = (
        [[float(b[0]), float(b[1]), float(b[2]), float(b[3])]
         for b in res_b.boxes.xyxy.cpu().numpy()]
        if res_b.boxes is not None and len(res_b.boxes) > 0 else []
    )

    # SAHI
    res_s = get_sliced_prediction(
        str(img_path), sahi_model,
        slice_height=SLICE_H, slice_width=SLICE_W,
        overlap_height_ratio=OVERLAP_RATIO, overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )
    preds_s = [[p.bbox.minx, p.bbox.miny, p.bbox.maxx, p.bbox.maxy]
               for p in res_s.object_prediction_list]

    for col, (preds, title) in enumerate([(preds_b, "Baseline"), (preds_s, "SAHI")]):
        img_draw = Image.fromarray(img_rgb)
        img_draw = draw_boxes_pil(img_draw, gts,   "#00FF00", 3)  # GT = vert
        img_draw = draw_boxes_pil(img_draw, preds, "#FF3300", 2)  # Pred = rouge
        tp, fp, fn = match_predictions(preds, gts)
        axes[idx][col].imshow(img_draw)
        axes[idx][col].set_title(
            f"{title} — {len(preds)} détections  (GT={len(gts)}, TP={tp}, FP={fp}, FN={fn})"
        )
        axes[idx][col].axis("off")

gt_patch   = mpatches.Patch(color="#00FF00", label="Ground truth")
pred_patch = mpatches.Patch(color="#FF3300", label="Prédictions")
fig.legend(handles=[gt_patch, pred_patch], loc="upper center", ncol=2, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8. SWEEP DES PARAMÈTRES DE TUILE ─────────────────────────
# Lance le sweep sur un sous-ensemble pour aller vite,
# puis utilise la meilleure config dans les cellules suivantes.
import time

SWEEP_N = 30   # nombre d'images pour le sweep (augmente si tu as le temps)

sweep_configs = [
    # (slice_h, slice_w, overlap)  — du plus fin au plus grossier
    (320,  320,  0.2),
    (320,  320,  0.3),
    (640,  640,  0.1),
    (640,  640,  0.2),   # point de départ recommandé
    (640,  640,  0.3),
    (960,  960,  0.2),
    (1280, 720,  0.2),   # ratio proche 16:9
]

sample_imgs = images[:SWEEP_N]
sweep_results = []

print(f"Sweep sur {len(sample_imgs)} images — {len(sweep_configs)} configs...\n")
print(f"{'Config':<22} {'P':>6} {'R':>6} {'F1':>6} {'temps':>8}")
print("-" * 55)

for sh, sw, ov in sweep_configs:
    tp_s = fp_s = fn_s = 0
    t0 = time.time()
    for img_path in sample_imgs:
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        res = get_sliced_prediction(
            str(img_path), sahi_model,
            slice_height=sh, slice_width=sw,
            overlap_height_ratio=ov, overlap_width_ratio=ov,
            verbose=0,
        )
        preds = [[p.bbox.minx, p.bbox.miny, p.bbox.maxx, p.bbox.maxy]
                 for p in res.object_prediction_list]
        gts = load_yolo_labels(
            str(Path(VAL_LABELS_DIR) / (img_path.stem + ".txt")), w, h
        )
        tp, fp, fn = match_predictions(preds, gts)
        tp_s += tp; fp_s += fp; fn_s += fn
    elapsed = time.time() - t0
    p, r, f1 = compute_metrics(tp_s, fp_s, fn_s)
    sweep_results.append((sh, sw, ov, p, r, f1))
    print(f"{sh}×{sw} ov={ov:<5} {p:>6.3f} {r:>6.3f} {f1:>6.3f} {elapsed:>6.1f}s")

best = max(sweep_results, key=lambda x: x[5])
print(f"\n★ Meilleure config (F1 max) : {best[0]}×{best[1]}, overlap={best[2]}")
print(f"  P={best[3]:.3f}  R={best[4]:.3f}  F1={best[5]:.3f}")
print("\nMets à jour SLICE_H, SLICE_W, OVERLAP_RATIO dans la cellule CONFIG si tu relances.")

In [ ]:
# ── 9. EXPORT VIDÉO ANNOTÉE (optionnel) ──────────────────────
# Exécute cette cellule séparément — elle peut prendre du temps.
# Utilise les paramètres SAHI de la cellule CONFIG.
import time

if not os.path.exists(VIDEO_PATH):
    print(f"Vidéo non trouvée : {VIDEO_PATH}\nAdapte VIDEO_PATH dans CONFIG.")
else:
    cap    = cv2.VideoCapture(VIDEO_PATH)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out    = cv2.VideoWriter(VIDEO_OUT_PATH, fourcc, fps, (width, height))

    print(f"Vidéo : {width}×{height}  {fps:.1f} fps  {total} frames")
    print(f"Sortie : {VIDEO_OUT_PATH}")

    frame_idx = 0
    t0 = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # SAHI attend du RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        result = get_sliced_prediction(
            frame_rgb,
            sahi_model,
            slice_height=SLICE_H,
            slice_width=SLICE_W,
            overlap_height_ratio=OVERLAP_RATIO,
            overlap_width_ratio=OVERLAP_RATIO,
            verbose=0,
        )

        for pred in result.object_prediction_list:
            b = pred.bbox
            x1, y1, x2, y2 = int(b.minx), int(b.miny), int(b.maxx), int(b.maxy)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                frame, f"{pred.score.value:.2f}",
                (x1, max(y1 - 5, 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1,
            )

        out.write(frame)
        frame_idx += 1

        if frame_idx % 50 == 0:
            elapsed = time.time() - t0
            pct = 100 * frame_idx / total
            eta = (elapsed / frame_idx) * (total - frame_idx)
            print(f"  {frame_idx}/{total} ({pct:.1f}%)  {frame_idx/elapsed:.1f} fps  ETA {eta:.0f}s")

    cap.release()
    out.release()
    print(f"\n✓ Vidéo annotée sauvegardée : {VIDEO_OUT_PATH}")